# Poaching Activity Prediction - Colab Pipeline
Upload your datasets to Colab and run cells sequentially.

In [ ]:
# Install dependencies
!pip install pandas numpy shapely scikit-learn imbalanced-learn xgboost matplotlib folium joblib seaborn branca -q

In [ ]:
# Mount Google Drive (optional: store datasets/outputs)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import pandas as pd
import numpy as np
from shapely.geometry import shape, Point, box
from shapely.prepared import prep
from shapely.ops import nearest_points, unary_union
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

# Configure paths - EDIT THESE to your dataset locations
DATA_CONFIG = {
    'park_boundary': '/content/drive/MyDrive/poaching_data/park_boundary.geojson',
    'roads': '/content/drive/MyDrive/poaching_data/roads.geojson',
    'villages': '/content/drive/MyDrive/poaching_data/villages.geojson',
    'water': '/content/drive/MyDrive/poaching_data/water.geojson',
    'poaching_incidents': '/content/drive/MyDrive/poaching_data/poaching_incidents.csv',
    'rainfall': '/content/drive/MyDrive/poaching_data/rainfall.csv',
    'moon_phases': '/content/drive/MyDrive/poaching_data/moon_phases.csv',
}

print("✓ Imports and config ready")

In [ ]:
# Load data utilities
def load_geojson_geoms(path):
    with open(path, 'r') as f:
        coll = json.load(f)
    res = []
    for feat in coll.get('features', []):
        geom = shape(feat['geometry'])
        props = feat.get('properties', {})
        res.append((geom, props))
    return res

def load_poaching(path):
    df = pd.read_csv(path, parse_dates=['date'])
    df['geometry'] = df.apply(lambda r: Point(r['lon'], r['lat']), axis=1)
    return df

print("✓ Data loaders defined")

In [ ]:

# ===== ALTERNATIVE: Direct File Upload (if Google Drive fails) =====
print("=== ALTERNATIVE DATA LOADING ===")
print("If Google Drive paths fail, use this method:")
print("\nFrom Colab file menu → Upload to session (click folder icon on left)")
print("Then update DATA_CONFIG paths to point to /content/uploaded files\n")

from google.colab import files
print("To upload files now, uncomment and run this:")
print("# files.upload()  # Select .geojson and .csv files")


In [ ]:
# Load all datasets
print("Loading datasets...")
try:
    park_list = load_geojson_geoms(DATA_CONFIG['park_boundary'])
    park_geom = park_list[0][0]
    roads = [g for g, p in load_geojson_geoms(DATA_CONFIG['roads'])]
    villages = [g for g, p in load_geojson_geoms(DATA_CONFIG['villages'])]
    water = [g for g, p in load_geojson_geoms(DATA_CONFIG['water'])]
    incidents = load_poaching(DATA_CONFIG['poaching_incidents'])
    rainfall_df = pd.read_csv(DATA_CONFIG['rainfall'], parse_dates=['week_start'])
    moon_df = pd.read_csv(DATA_CONFIG['moon_phases'], parse_dates=['date'])
    
    print(f"✓ Park boundary loaded")
    print(f"✓ Roads: {len(roads)} features")
    print(f"✓ Villages: {len(villages)} features")
    print(f"✓ Water: {len(water)} features")
    print(f"✓ Poaching incidents: {len(incidents)} records")
    print(f"✓ Rainfall: {len(rainfall_df)} weeks")
    print(f"✓ Moon phases: {len(moon_df)} dates")
except Exception as e:
    print(f"✗ Error loading data: {e}")
    print("Check DATA_CONFIG paths above.")

In [ ]:
# Create grid
def create_grid(park_geom, cell_size_deg=0.01):
    minx, miny, maxx, maxy = park_geom.bounds
    x_coords = np.arange(minx, maxx, cell_size_deg)
    y_coords = np.arange(miny, maxy, cell_size_deg)
    polys = []
    ids = []
    for i, x in enumerate(x_coords):
        for j, y in enumerate(y_coords):
            poly = box(x, y, x+cell_size_deg, y+cell_size_deg)
            if poly.intersects(park_geom):
                polys.append(poly.intersection(park_geom))
                ids.append(f"{i}_{j}")
    return pd.DataFrame({'grid_id':ids, 'geometry':polys})

print("Creating spatial grid...")
grid = create_grid(park_geom, cell_size_deg=0.01)
print(f"✓ Grid created: {len(grid)} cells")

In [ ]:
# Load all datasets with error handling and fallback
import os
from pathlib import Path

print("Loading datasets from configured paths...")
failed_files = []
loaded_files = []

for key, path in DATA_CONFIG.items():
    if os.path.exists(path):
        loaded_files.append(f"✓ {key}: {path}")
    else:
        failed_files.append(f"✗ {key}: NOT FOUND at {path}")

print("\n" + "\n".join(loaded_files))

if failed_files:
    print("\n⚠ FAILED TO FIND:")
    print("\n".join(failed_files))
    print("\nFIX OPTIONS:")
    print("1. Check paths are correct in DATA_CONFIG (edit cell 3)")
    print("2. Check folder exists: /content/drive/MyDrive/poaching_data/")
    print("3. OR upload files directly and update paths to /content/")
    print("4. OR check file permissions in Google Drive")
else:
    try:
        park_list = load_geojson_geoms(DATA_CONFIG['park_boundary'])
        park_geom = park_list[0][0]
        roads = [g for g, p in load_geojson_geoms(DATA_CONFIG['roads'])]
        villages = [g for g, p in load_geojson_geoms(DATA_CONFIG['villages'])]
        water = [g for g, p in load_geojson_geoms(DATA_CONFIG['water'])]
        incidents = load_poaching(DATA_CONFIG['poaching_incidents'])
        rainfall_df = pd.read_csv(DATA_CONFIG['rainfall'], parse_dates=['week_start'])
        moon_df = pd.read_csv(DATA_CONFIG['moon_phases'], parse_dates=['date'])
        
        print(f"\n✓ Park boundary loaded")
        print(f"✓ Roads: {len(roads)} features")
        print(f"✓ Villages: {len(villages)} features")
        print(f"✓ Water: {len(water)} features")
        print(f"✓ Poaching incidents: {len(incidents)} records")
        print(f"✓ Rainfall: {len(rainfall_df)} weeks")
        print(f"✓ Moon phases: {len(moon_df)} dates")
    except Exception as e:
        print(f"\n✗ Error loading data: {e}")
        print("\nTroubleshooting:")
        print("- Check dataset format (GeoJSON for .geojson, CSV for .csv)")
        print("- Check CSV has correct columns: date, lon, lat (for incidents)")
        print("- Check GeoJSON is valid JSON format")
        print("- Try uploading files directly: files.upload()")


In [ ]:
# Aggregate weekly
print("Aggregating data by week...")
assigned_valid['week_start'] = assigned_valid['date'].dt.to_period('W').apply(lambda r: r.start_time)
weekly = assigned_valid.groupby(['grid_id','week_start']).size().reset_index(name='poaching_count')

# Full grid-week combinations
weeks = pd.date_range(start=assigned_valid['date'].min(), periods=52, freq='W-MON')
base = pd.MultiIndex.from_product([grid['grid_id'], weeks], names=['grid_id','week_start']).to_frame(index=False)
weekly_full = base.merge(weekly, on=['grid_id','week_start'], how='left').fillna({'poaching_count':0})
print(f"✓ Weekly aggregation: {len(weekly_full)} records ({len(grid)} grids × {len(weeks)} weeks)")

In [ ]:
# Compute distance features
def compute_distance_to_nearest(grid_df, target_geoms, col_name):
    try:
        targ_union = unary_union(target_geoms)
    except:
        targ_union = target_geoms[0] if target_geoms else None
    dists = []
    for geom in grid_df['geometry']:
        cent = geom.centroid
        if targ_union is None:
            dists.append(np.nan)
        else:
            near = nearest_points(cent, targ_union)[1]
            dists.append(cent.distance(near))
    grid_df[col_name] = dists
    return grid_df

print("Computing static distance features...")
grid = compute_distance_to_nearest(grid, roads, 'dist_road')
grid = compute_distance_to_nearest(grid, villages, 'dist_village')
grid = compute_distance_to_nearest(grid, water, 'dist_water')
print("✓ Distance features computed")

In [ ]:
# NDVI feature (synthetic based on location)
def mean_ndvi_per_grid(grid_df, seed=0):
    rng = np.random.default_rng(seed)
    vals = []
    for geom in grid_df['geometry']:
        cent = geom.centroid
        val = 0.4 + 0.2 * np.sin(cent.x * 10) + 0.1 * np.cos(cent.y * 12)
        val += rng.normal(0, 0.05)
        vals.append(float(np.clip(val, -1, 1)))
    grid_df['mean_ndvi'] = vals
    return grid_df

grid = mean_ndvi_per_grid(grid, seed=0)
print("✓ NDVI feature added")

In [ ]:
# Merge static features into weekly
print("Merging static features...")
weekly_full = weekly_full.merge(grid[['grid_id','dist_road','dist_village','dist_water','mean_ndvi']], on='grid_id', how='left')
weekly_full['target'] = (weekly_full['poaching_count'] > 0).astype(int)

# Add lag feature
weekly_full = weekly_full.sort_values(['grid_id','week_start'])
weekly_full['poaching_lag'] = weekly_full.groupby('grid_id')['poaching_count'].shift(1).fillna(0)
print(f"✓ Features merged. Class balance: {weekly_full['target'].value_counts().to_dict()}")

In [ ]:
# Prepare data for training
feature_cols = ['dist_road','dist_village','dist_water','mean_ndvi','poaching_lag']
X = weekly_full[feature_cols].fillna(0)
y = weekly_full['target']

print(f"✓ Data prepared: X shape {X.shape}, y shape {y.shape}")
print(f"  Feature columns: {feature_cols}")
print(f"  Class distribution: {y.value_counts().to_dict()}")

In [ ]:
# Train models
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, confusion_matrix
from imblearn.over_sampling import SMOTE
from joblib import dump

print("Training models...")

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"  Train: {len(X_train)}, Test: {len(X_test)}")

# SMOTE for imbalance
sm = SMOTE(random_state=42, k_neighbors=3)
X_res, y_res = sm.fit_resample(X_train, y_train)
print(f"  After SMOTE: {len(X_res)}")

# Random Forest
print("  Training Random Forest...")
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_params = {'n_estimators':[100,200], 'max_depth':[5,10]}
rf_cv = RandomizedSearchCV(rf, rf_params, n_iter=4, cv=3, scoring='roc_auc', random_state=42, n_jobs=-1)
rf_cv.fit(X_res, y_res)
print(f"    Best params: {rf_cv.best_params_}")

# XGBoost
print("  Training XGBoost...")
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)
xgb_params = {'n_estimators':[100,200], 'max_depth':[3,6], 'learning_rate':[0.05,0.1]}
xgb_cv = RandomizedSearchCV(xgb, xgb_params, n_iter=4, cv=3, scoring='roc_auc', random_state=42, n_jobs=-1)
xgb_cv.fit(X_res, y_res)
print(f"    Best params: {xgb_cv.best_params_}")

In [ ]:
# Evaluate ensemble
print("Evaluating ensemble...")

p1_rf = rf_cv.predict_proba(X_test)[:,1]
p1_xgb = xgb_cv.predict_proba(X_test)[:,1]
p_ensemble = (p1_rf + p1_xgb) / 2.0

auc_rf = roc_auc_score(y_test, p1_rf)
auc_xgb = roc_auc_score(y_test, p1_xgb)
auc_ens = roc_auc_score(y_test, p_ensemble)

print(f"✓ Random Forest AUC: {auc_rf:.4f}")
print(f"✓ XGBoost AUC: {auc_xgb:.4f}")
print(f"✓ Ensemble AUC: {auc_ens:.4f}")

if auc_ens > 0.8:
    print("\n🎯 Excellent! AUC > 0.8 achieved.")
else:
    print(f"\n⚠ AUC is {auc_ens:.4f}. Consider more features or tuning.")

In [ ]:
# Attach probabilities
print("Generating risk probabilities...")
weekly_full['risk_proba'] = (rf_cv.predict_proba(X)[:,1] + xgb_cv.predict_proba(X)[:,1]) / 2.0

# Aggregate risk per grid
grid_risk = weekly_full.groupby('grid_id')['risk_proba'].mean().reset_index()
grid_out = grid.merge(grid_risk, on='grid_id', how='left')
grid_out['risk_proba'] = grid_out['risk_proba'].fillna(0)

print(f"✓ Risk probabilities computed")
print(f"  Min risk: {grid_out['risk_proba'].min():.4f}")
print(f"  Max risk: {grid_out['risk_proba'].max():.4f}")
print(f"  Mean risk: {grid_out['risk_proba'].mean():.4f}")

In [ ]:
# Save outputs to Google Drive
import os
output_dir = '/content/drive/MyDrive/poaching_outputs'
os.makedirs(output_dir, exist_ok=True)

# Save models
dump(rf_cv, f'{output_dir}/rf_cv.joblib')
dump(xgb_cv, f'{output_dir}/xgb_cv.joblib')

# Save grid with probabilities (for GIS)
grid_out_geojson = []
for _, row in grid_out.iterrows():
    from shapely.geometry import mapping
    feat = {
        "type": "Feature",
        "properties": {
            "grid_id": row['grid_id'],
            "risk_proba": float(row['risk_proba']),
            "dist_road": float(row['dist_road']),
            "dist_village": float(row['dist_village']),
            "dist_water": float(row['dist_water']),
            "mean_ndvi": float(row['mean_ndvi'])
        },
        "geometry": mapping(row['geometry'])
    }
    grid_out_geojson.append(feat)

with open(f'{output_dir}/grid_risk_output.geojson', 'w') as f:
    json.dump({"type": "FeatureCollection", "features": grid_out_geojson}, f)

# Save metrics
metrics = {
    'rf_auc': float(auc_rf),
    'xgb_auc': float(auc_xgb),
    'ensemble_auc': float(auc_ens),
    'total_grids': len(grid_out),
    'total_incidents': len(incidents),
}
with open(f'{output_dir}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"✓ Outputs saved to {output_dir}")
print("  - rf_cv.joblib (trained model)")
print("  - xgb_cv.joblib (trained model)")
print("  - grid_risk_output.geojson (GIS-ready risk heatmap)")
print("  - metrics.json (evaluation results)")

In [ ]:
# Display top 15 high-risk grids (patrol priority)
print("\n🚨 TOP 15 HIGH-RISK GRIDS (Patrol Priority):")
top_grids = grid_out.sort_values(by='risk_proba', ascending=False).head(15)[['grid_id', 'risk_proba', 'dist_road', 'dist_village']]
for idx, row in top_grids.iterrows():
    print(f"  {row['grid_id']}: risk={row['risk_proba']:.4f}, to_road={row['dist_road']:.4f}°, to_village={row['dist_village']:.4f}°")

# Save top grids as patrol CSV
top_grids.to_csv(f'{output_dir}/patrol_priority_zones.csv', index=False)
print(f"\n✓ Patrol priorities saved to patrol_priority_zones.csv")

## Next Steps
1. Download `grid_risk_output.geojson` and open in QGIS or ArcGIS for visualization
2. Use `patrol_priority_zones.csv` to plan ranger patrols
3. Export models and metrics for deployment or further analysis